## ROUND 5 ##
Check for:
- correlation between the asset prices in the same category
- adfuller test
- give each category a classification if possible


In [1]:
import sys
from pathlib import Path
root = Path.cwd().parent
sys.path.append(str(root))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant
from core.normalizer import compute_wallmid3

In [4]:
GALAXY_DM = "GALAXY_SOUNDS_DARK_MATTER"
GALAXY_BH = "GALAXY_SOUNDS_BLACK_HOLES"
GALAXY_PR = "GALAXY_SOUNDS_PLANETARY_RINGS"
GALAXY_SW = "GALAXY_SOUNDS_SOLAR_WINDS"
GALAXY_SF = "GALAXY_SOUNDS_SOLAR_FLAMES"

POD_SUEDE = "SLEEP_POD_SUEDE"
POD_LAMB = "SLEEP_POD_LAMB_WOOL"
POD_POLY = "SLEEP_POD_POLYESTER"
POD_NYLON = "SLEEP_POD_NYLON"
POD_COTTON = "SLEEP_POD_COTTON"

CHIP_CIRCLE = "MICROCHIP_CIRCLE"
CHIP_OVAL = "MICROCHIP_OVAL"
CHIP_SQUARE = "MICROCHIP_SQUARE"
CHIP_RECT = "MICROCHIP_RECTANGLE"
CHIP_TRI = "MICROCHIP_TRIANGLE"

PEB_XS = "PEBBLES_XS"
PEB_S = "PEBBLES_S"
PEB_M = "PEBBLES_M"
PEB_L = "PEBBLES_L"
PEB_XL = "PEBBLES_XL"

ROBOT_VAC = "ROBOT_VACUUMING"
ROBOT_MOP = "ROBOT_MOPPING"
ROBOT_DISH = "ROBOT_DISHES"
ROBOT_LAUN = "ROBOT_LAUNDRY"
ROBOT_IRON = "ROBOT_IRONING"

UV_YELLOW = "UV_VISOR_YELLOW"
UV_AMBER = "UV_VISOR_AMBER"
UV_ORANGE = "UV_VISOR_ORANGE"
UV_RED = "UV_VISOR_RED"
UV_MAGENTA = "UV_VISOR_MAGENTA"

TRANS_GRAY = "TRANSLATOR_SPACE_GRAY"
TRANS_BLACK = "TRANSLATOR_ASTRO_BLACK"
TRANS_CHAR = "TRANSLATOR_ECLIPSE_CHARCOAL"
TRANS_MIST = "TRANSLATOR_GRAPHITE_MIST"
TRANS_BLUE = "TRANSLATOR_VOID_BLUE"

PANEL_1X2 = "PANEL_1X2"
PANEL_2X2 = "PANEL_2X2"
PANEL_1X4 = "PANEL_1X4"
PANEL_2X4 = "PANEL_2X4"
PANEL_4X4 = "PANEL_4X4"

OXY_MORN = "OXYGEN_SHAKE_MORNING_BREATH"
OXY_EVE = "OXYGEN_SHAKE_EVENING_BREATH"
OXY_MINT = "OXYGEN_SHAKE_MINT"
OXY_CHOC = "OXYGEN_SHAKE_CHOCOLATE"
OXY_GARLIC = "OXYGEN_SHAKE_GARLIC"

SNACK_CHOC = "SNACKPACK_CHOCOLATE"
SNACK_VAN = "SNACKPACK_VANILLA"
SNACK_PIST = "SNACKPACK_PISTACHIO"
SNACK_STRAW = "SNACKPACK_STRAWBERRY"
SNACK_RASP = "SNACKPACK_RASPBERRY"

products = {
    "GALAXY_DM": GALAXY_DM, "GALAXY_BH": GALAXY_BH, "GALAXY_PR": GALAXY_PR,
    "GALAXY_SW": GALAXY_SW, "GALAXY_SF": GALAXY_SF,
    "POD_SUEDE": POD_SUEDE, "POD_LAMB": POD_LAMB, "POD_POLY": POD_POLY,
    "POD_NYLON": POD_NYLON, "POD_COTTON": POD_COTTON,
    "CHIP_CIRCLE": CHIP_CIRCLE, "CHIP_OVAL": CHIP_OVAL, "CHIP_SQUARE": CHIP_SQUARE,
    "CHIP_RECT": CHIP_RECT, "CHIP_TRI": CHIP_TRI,
    "PEB_XS": PEB_XS, "PEB_S": PEB_S, "PEB_M": PEB_M, "PEB_L": PEB_L, "PEB_XL": PEB_XL,
    "ROBOT_VAC": ROBOT_VAC, "ROBOT_MOP": ROBOT_MOP, "ROBOT_DISH": ROBOT_DISH,
    "ROBOT_LAUN": ROBOT_LAUN, "ROBOT_IRON": ROBOT_IRON,
    "UV_YELLOW": UV_YELLOW, "UV_AMBER": UV_AMBER, "UV_ORANGE": UV_ORANGE,
    "UV_RED": UV_RED, "UV_MAGENTA": UV_MAGENTA,
    "TRANS_GRAY": TRANS_GRAY, "TRANS_BLACK": TRANS_BLACK, "TRANS_CHAR": TRANS_CHAR,
    "TRANS_MIST": TRANS_MIST, "TRANS_BLUE": TRANS_BLUE,
    "PANEL_1X2": PANEL_1X2, "PANEL_2X2": PANEL_2X2, "PANEL_1X4": PANEL_1X4,
    "PANEL_2X4": PANEL_2X4, "PANEL_4X4": PANEL_4X4,
    "OXY_MORN": OXY_MORN, "OXY_EVE": OXY_EVE, "OXY_MINT": OXY_MINT,
    "OXY_CHOC": OXY_CHOC, "OXY_GARLIC": OXY_GARLIC,
    "SNACK_CHOC": SNACK_CHOC, "SNACK_VAN": SNACK_VAN, "SNACK_PIST": SNACK_PIST,
    "SNACK_STRAW": SNACK_STRAW, "SNACK_RASP": SNACK_RASP,
}

path = "/Users/alexoddsh/prosperity/backend/backtester/resources-4/round5/"

price_files = ["prices_round_5_day_2.csv", "prices_round_5_day_3.csv", "prices_round_5_day_4.csv"]
trade_files = ["trades_round_5_day_2.csv", "trades_round_5_day_3.csv", "trades_round_5_day_4.csv"]

df_list = [pd.read_csv(path + f, sep=";") for f in price_files]
tf_list = [pd.read_csv(path + f, sep=";") for f in trade_files]

all_dfs: dict[str, pd.DataFrame] = {}
all_tfs: dict[str, pd.DataFrame] = {}

for key, symbol in products.items():
    p_days = []
    for i, df in enumerate(df_list):
        sub = df[df["product"] == symbol].copy()
        if i > 0:
            sub["timestamp"] += i * 1000000
        sub["mid_price"] = sub["mid_price"].ffill()
        p_days.append(sub)

    combined_df = pd.concat(p_days)

    fill_cols = ["ask_price_1", "bid_price_1", "ask_price_2", "bid_price_2", "ask_price_3", "bid_price_3"]
    existing = [c for c in fill_cols if c in combined_df.columns]
    for col in existing:
        combined_df[col] = combined_df[col].ffill()

    all_dfs[key] = pd.DataFrame(combined_df)

    t_days = []
    for i, tf in enumerate(tf_list):
        sub_t = tf[tf["symbol"] == symbol].copy()
        if i > 0:
            sub_t["timestamp"] += i * 1000000
        t_days.append(sub_t)

    combined_tf = pd.concat(t_days)
    all_tfs[key] = pd.DataFrame(combined_tf)


df_galaxy_dm: pd.DataFrame = all_dfs["GALAXY_DM"]
df_galaxy_bh: pd.DataFrame = all_dfs["GALAXY_BH"]
df_galaxy_pr: pd.DataFrame = all_dfs["GALAXY_PR"]
df_galaxy_sw: pd.DataFrame = all_dfs["GALAXY_SW"]
df_galaxy_sf: pd.DataFrame = all_dfs["GALAXY_SF"]
df_pod_suede: pd.DataFrame = all_dfs["POD_SUEDE"]
df_pod_lamb: pd.DataFrame = all_dfs["POD_LAMB"]
df_pod_poly: pd.DataFrame = all_dfs["POD_POLY"]
df_pod_nylon: pd.DataFrame = all_dfs["POD_NYLON"]
df_pod_cotton: pd.DataFrame = all_dfs["POD_COTTON"]
df_chip_circle: pd.DataFrame = all_dfs["CHIP_CIRCLE"]
df_chip_oval: pd.DataFrame = all_dfs["CHIP_OVAL"]
df_chip_square: pd.DataFrame = all_dfs["CHIP_SQUARE"]
df_chip_rect: pd.DataFrame = all_dfs["CHIP_RECT"]
df_chip_tri: pd.DataFrame = all_dfs["CHIP_TRI"]
df_peb_xs: pd.DataFrame = all_dfs["PEB_XS"]
df_peb_s: pd.DataFrame = all_dfs["PEB_S"]
df_peb_m: pd.DataFrame = all_dfs["PEB_M"]
df_peb_l: pd.DataFrame = all_dfs["PEB_L"]
df_peb_xl: pd.DataFrame = all_dfs["PEB_XL"]
df_robot_vac: pd.DataFrame = all_dfs["ROBOT_VAC"]
df_robot_mop: pd.DataFrame = all_dfs["ROBOT_MOP"]
df_robot_dish: pd.DataFrame = all_dfs["ROBOT_DISH"]
df_robot_laun: pd.DataFrame = all_dfs["ROBOT_LAUN"]
df_robot_iron: pd.DataFrame = all_dfs["ROBOT_IRON"]
df_uv_yellow: pd.DataFrame = all_dfs["UV_YELLOW"]
df_uv_amber: pd.DataFrame = all_dfs["UV_AMBER"]
df_uv_orange: pd.DataFrame = all_dfs["UV_ORANGE"]
df_uv_red: pd.DataFrame = all_dfs["UV_RED"]
df_uv_magenta: pd.DataFrame = all_dfs["UV_MAGENTA"]
df_trans_gray: pd.DataFrame = all_dfs["TRANS_GRAY"]
df_trans_black: pd.DataFrame = all_dfs["TRANS_BLACK"]
df_trans_char: pd.DataFrame = all_dfs["TRANS_CHAR"]
df_trans_mist: pd.DataFrame = all_dfs["TRANS_MIST"]
df_trans_blue: pd.DataFrame = all_dfs["TRANS_BLUE"]
df_panel_1x2: pd.DataFrame = all_dfs["PANEL_1X2"]
df_panel_2x2: pd.DataFrame = all_dfs["PANEL_2X2"]
df_panel_1x4: pd.DataFrame = all_dfs["PANEL_1X4"]
df_panel_2x4: pd.DataFrame = all_dfs["PANEL_2X4"]
df_panel_4x4: pd.DataFrame = all_dfs["PANEL_4X4"]
df_oxy_morn: pd.DataFrame = all_dfs["OXY_MORN"]
df_oxy_eve: pd.DataFrame = all_dfs["OXY_EVE"]
df_oxy_mint: pd.DataFrame = all_dfs["OXY_MINT"]
df_oxy_choc: pd.DataFrame = all_dfs["OXY_CHOC"]
df_oxy_garlic: pd.DataFrame = all_dfs["OXY_GARLIC"]
df_snack_choc: pd.DataFrame = all_dfs["SNACK_CHOC"]
df_snack_van: pd.DataFrame = all_dfs["SNACK_VAN"]
df_snack_pist: pd.DataFrame = all_dfs["SNACK_PIST"]
df_snack_straw: pd.DataFrame = all_dfs["SNACK_STRAW"]
df_snack_rasp: pd.DataFrame = all_dfs["SNACK_RASP"]

tf_galaxy_dm: pd.DataFrame = all_tfs["GALAXY_DM"]
tf_galaxy_bh: pd.DataFrame = all_tfs["GALAXY_BH"]
tf_galaxy_pr: pd.DataFrame = all_tfs["GALAXY_PR"]
tf_galaxy_sw: pd.DataFrame = all_tfs["GALAXY_SW"]
tf_galaxy_sf: pd.DataFrame = all_tfs["GALAXY_SF"]
tf_pod_suede: pd.DataFrame = all_tfs["POD_SUEDE"]
tf_pod_lamb: pd.DataFrame = all_tfs["POD_LAMB"]
tf_pod_poly: pd.DataFrame = all_tfs["POD_POLY"]
tf_pod_nylon: pd.DataFrame = all_tfs["POD_NYLON"]
tf_pod_cotton: pd.DataFrame = all_tfs["POD_COTTON"]
tf_chip_circle: pd.DataFrame = all_tfs["CHIP_CIRCLE"]
tf_chip_oval: pd.DataFrame = all_tfs["CHIP_OVAL"]
tf_chip_square: pd.DataFrame = all_tfs["CHIP_SQUARE"]
tf_chip_rect: pd.DataFrame = all_tfs["CHIP_RECT"]
tf_chip_tri: pd.DataFrame = all_tfs["CHIP_TRI"]
tf_peb_xs: pd.DataFrame = all_tfs["PEB_XS"]
tf_peb_s: pd.DataFrame = all_tfs["PEB_S"]
tf_peb_m: pd.DataFrame = all_tfs["PEB_M"]
tf_peb_l: pd.DataFrame = all_tfs["PEB_L"]
tf_peb_xl: pd.DataFrame = all_tfs["PEB_XL"]
tf_robot_vac: pd.DataFrame = all_tfs["ROBOT_VAC"]
tf_robot_mop: pd.DataFrame = all_tfs["ROBOT_MOP"]
tf_robot_dish: pd.DataFrame = all_tfs["ROBOT_DISH"]
tf_robot_laun: pd.DataFrame = all_tfs["ROBOT_LAUN"]
tf_robot_iron: pd.DataFrame = all_tfs["ROBOT_IRON"]
tf_uv_yellow: pd.DataFrame = all_tfs["UV_YELLOW"]
tf_uv_amber: pd.DataFrame = all_tfs["UV_AMBER"]
tf_uv_orange: pd.DataFrame = all_tfs["UV_ORANGE"]
tf_uv_red: pd.DataFrame = all_tfs["UV_RED"]
tf_uv_magenta: pd.DataFrame = all_tfs["UV_MAGENTA"]
tf_trans_gray: pd.DataFrame = all_tfs["TRANS_GRAY"]
tf_trans_black: pd.DataFrame = all_tfs["TRANS_BLACK"]
tf_trans_char: pd.DataFrame = all_tfs["TRANS_CHAR"]
tf_trans_mist: pd.DataFrame = all_tfs["TRANS_MIST"]
tf_trans_blue: pd.DataFrame = all_tfs["TRANS_BLUE"]
tf_panel_1x2: pd.DataFrame = all_tfs["PANEL_1X2"]
tf_panel_2x2: pd.DataFrame = all_tfs["PANEL_2X2"]
tf_panel_1x4: pd.DataFrame = all_tfs["PANEL_1X4"]
tf_panel_2x4: pd.DataFrame = all_tfs["PANEL_2X4"]
tf_panel_4x4: pd.DataFrame = all_tfs["PANEL_4X4"]
tf_oxy_morn: pd.DataFrame = all_tfs["OXY_MORN"]
tf_oxy_eve: pd.DataFrame = all_tfs["OXY_EVE"]
tf_oxy_mint: pd.DataFrame = all_tfs["OXY_MINT"]
tf_oxy_choc: pd.DataFrame = all_tfs["OXY_CHOC"]
tf_oxy_garlic: pd.DataFrame = all_tfs["OXY_GARLIC"]
tf_snack_choc: pd.DataFrame = all_tfs["SNACK_CHOC"]
tf_snack_van: pd.DataFrame = all_tfs["SNACK_VAN"]
tf_snack_pist: pd.DataFrame = all_tfs["SNACK_PIST"]
tf_snack_straw: pd.DataFrame = all_tfs["SNACK_STRAW"]
tf_snack_rasp: pd.DataFrame = all_tfs["SNACK_RASP"]

categories: dict[str, list[str]] = {
    "GALAXY_SOUNDS": ["GALAXY_DM", "GALAXY_BH", "GALAXY_PR", "GALAXY_SW", "GALAXY_SF"],
    "SLEEP_PODS":    ["POD_SUEDE", "POD_LAMB", "POD_POLY", "POD_NYLON", "POD_COTTON"],
    "MICROCHIPS":    ["CHIP_CIRCLE", "CHIP_OVAL", "CHIP_SQUARE", "CHIP_RECT", "CHIP_TRI"],
    "PEBBLES":       ["PEB_XS", "PEB_S", "PEB_M", "PEB_L", "PEB_XL"],
    "ROBOTS":        ["ROBOT_VAC", "ROBOT_MOP", "ROBOT_DISH", "ROBOT_LAUN", "ROBOT_IRON"],
    "UV_VISORS":     ["UV_YELLOW", "UV_AMBER", "UV_ORANGE", "UV_RED", "UV_MAGENTA"],
    "TRANSLATORS":   ["TRANS_GRAY", "TRANS_BLACK", "TRANS_CHAR", "TRANS_MIST", "TRANS_BLUE"],
    "PANELS":        ["PANEL_1X2", "PANEL_2X2", "PANEL_1X4", "PANEL_2X4", "PANEL_4X4"],
    "OXYGEN_SHAKES": ["OXY_MORN", "OXY_EVE", "OXY_MINT", "OXY_CHOC", "OXY_GARLIC"],
    "SNACKPACKS":    ["SNACK_CHOC", "SNACK_VAN", "SNACK_PIST", "SNACK_STRAW", "SNACK_RASP"],
}

In [14]:
SELECTED_CATEGORY = "SNACKPACKS"

for key in categories[SELECTED_CATEGORY]:
    symbol = products[key]
    df = all_dfs[key]

    compute_wallmid3(symbol, df)
    wallmid = df["wallmid3"].ffill().bfill().to_numpy()
    null_reject = False

    # regression "c" -> variant 2 and "ct" = variant 3
    # c for constant, t for trend
    # autolag=None means not augmented
    t_stat, p_val, usedlag, nobs, critical_values = adfuller(  # type: ignore
        wallmid, maxlag=0, regression="c", autolag=None
    )

    if t_stat < critical_values["5%"]:
        null_reject = True

    print(f"\n===== {symbol} ({key}) =====")

    if null_reject:
        delta_y = np.diff(wallmid)
        y_lag = wallmid[:-1]
        X = add_constant(y_lag)
        ols_res = OLS(delta_y, X).fit()
        intercept = ols_res.params[0]
        rho = ols_res.params[1]
        phi = rho + 1

        half_life = np.log(2) / -np.log(phi)

        mu_wallmid = wallmid.mean()
        sigma_wallmid = wallmid.std(ddof=1)  
        z_scores = (wallmid - mu_wallmid) / sigma_wallmid

        above = np.abs(z_scores) > 0.75
        crossings = np.diff(above.astype(int))
        entry_crossings = np.where(crossings == 1)[0] + 1
        entry_signals = int(np.sum(crossings == 1))

        print(f"NULL HYPOTHESIS REJECTED: {null_reject}")
        print(f"RHO: {rho}")
        print(f"PHI: {phi}")
        print(f"REG EQUILIBRIUM: {-intercept / rho}")
        print(f"T-STAT: {t_stat}")
        print(f"P-VAL: {p_val}")
        print(f"Half Life: {half_life}")
        print(f"Mean Wallmid: {mu_wallmid}")
        print(f"Std. Wallmid: {sigma_wallmid}")
        print(f"Z-Score Signals: {entry_signals}")
    else:
        print(f"NULL HYPOTHESIS REJECTED: {null_reject}")
        print(f"T-STAT: {t_stat}")
        print(f"P-VAL: {p_val}")


===== SNACKPACK_CHOCOLATE (SNACK_CHOC) =====
NULL HYPOTHESIS REJECTED: False
T-STAT: -2.0296861365406094
P-VAL: 0.2737103273573022

===== SNACKPACK_VANILLA (SNACK_VAN) =====
NULL HYPOTHESIS REJECTED: False
T-STAT: -2.2259937879546237
P-VAL: 0.196946821596525

===== SNACKPACK_PISTACHIO (SNACK_PIST) =====
NULL HYPOTHESIS REJECTED: False
T-STAT: -2.2016770489129835
P-VAL: 0.20563734800305644

===== SNACKPACK_STRAWBERRY (SNACK_STRAW) =====
NULL HYPOTHESIS REJECTED: False
T-STAT: -2.0386334754439894
P-VAL: 0.2698946673874115

===== SNACKPACK_RASPBERRY (SNACK_RASP) =====
NULL HYPOTHESIS REJECTED: True
RHO: -0.000600673683787162
PHI: 0.9993993263162129
REG EQUILIBRIUM: 10097.291340403679
T-STAT: -2.9139205958706462
P-VAL: 0.04375300401961497
Half Life: 1153.6030306979983
Mean Wallmid: 10078.376266666666
Std. Wallmid: 168.59646877195155
Z-Score Signals: 205


In [16]:
SELECTED_CATEGORY = "SNACKPACKS"

print(f"\n===== Avg Spread - {SELECTED_CATEGORY} =====")
for key in categories[SELECTED_CATEGORY]:
    symbol = products[key]
    df = all_dfs[key]

    spread = df["ask_price_1"].ffill().bfill() - df["bid_price_1"].ffill().bfill()
    avg_spread = spread.mean()

    print(f"{symbol:<35} {avg_spread:.4f}")


===== Avg Spread - SNACKPACKS =====
SNACKPACK_CHOCOLATE                 16.4712
SNACKPACK_VANILLA                   16.8687
SNACKPACK_PISTACHIO                 15.9256
SNACKPACK_STRAWBERRY                17.8265
SNACKPACK_RASPBERRY                 16.8425
